In [1]:
# ============================================================
#  Distri-Cluster: GPU K-Means (CUDA via PyTorch)
#  Kaggle T4 notebook — matches vectors.bin binary format exactly
#  Drop this entire file into a Kaggle notebook cell and run.
# ============================================================
#
#  SETUP INSTRUCTIONS:
#  1. On Kaggle: File > Upload Dataset > upload your vectors.bin
#  2. In notebook settings: Accelerator = GPU T4 x1
#  3. Paste this into a code cell and run
#  4. Copy the printed timing line into your benchmark_results.csv
# ============================================================

import struct
import time
import numpy as np
import torch

# ── Config (must match your existing experiments exactly) ────────────────────
K         = 10
MAX_ITER  = 50
SEED      = 42

# ── Path: update this to wherever Kaggle mounts your vectors.bin ─────────────
# If you uploaded via Kaggle Datasets, it will be something like:
#   /kaggle/input/<dataset-name>/vectors.bin
# If you uploaded as a notebook input file directly:
#   /kaggle/input/vectors.bin
VECTORS_BIN = "/kaggle/input/datasets/sadhanadevarajan/cifar-10-vectors/vectors.bin"

# ── 1. Load vectors.bin (exact same format as your C++ code) ─────────────────
print("Loading vectors.bin ...")
with open(VECTORS_BIN, "rb") as f:
    N = struct.unpack("i", f.read(4))[0]
    D = struct.unpack("i", f.read(4))[0]
    data = np.frombuffer(f.read(N * D * 4), dtype=np.float32).reshape(N, D)

print(f"Loaded {N} vectors, {D} dims")
print(f"K={K}  MAX_ITER={MAX_ITER}  SEED={SEED}")

# ── 2. Move data to GPU ───────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type != "cuda":
    raise RuntimeError("No GPU found — make sure Kaggle accelerator is set to GPU T4 x1")

X = torch.tensor(data, dtype=torch.float32, device=device)  # (N, D)

# ── 3. K-Means++ initialisation (same logic as your C++ kmeans_pp) ───────────
# Must use same seed=42 to get comparable (not identical) initialisation.
# Note: results won't be bit-identical to CPU (different FP order on GPU)
# but inertia and cluster quality will be equivalent.

torch.manual_seed(SEED)
np.random.seed(SEED)

print("K-Means++ initialisation ...")

def kmeans_plus_plus_init(X, K, seed=42):
    """
    Exact K-Means++ logic matching your C++ init_centroids_pp().
    Selects first centroid randomly, then each subsequent centroid
    with probability proportional to squared distance from nearest
    already-chosen centroid.
    """
    N = X.shape[0]
    torch.manual_seed(seed)

    # Pick first centroid randomly
    idx = torch.randint(0, N, (1,)).item()
    centroids = [X[idx]]

    for _ in range(1, K):
        # Stack current centroids: (c, D)
        C = torch.stack(centroids)  # (c, D)

        # Compute squared distances from all points to nearest centroid
        # X: (N, D), C: (c, D)
        # dist[i] = min over c of ||X[i] - C[c]||^2
        diffs = X.unsqueeze(1) - C.unsqueeze(0)   # (N, c, D)
        sq_dists = (diffs ** 2).sum(dim=2)          # (N, c)
        min_sq_dists = sq_dists.min(dim=1).values   # (N,)

        # Sample next centroid proportional to squared distance
        probs = min_sq_dists / min_sq_dists.sum()
        probs_cpu = probs.cpu().numpy()
        next_idx = np.random.choice(N, p=probs_cpu)
        centroids.append(X[next_idx])

    return torch.stack(centroids)  # (K, D)

centroids = kmeans_plus_plus_init(X, K, seed=SEED)
print("K-Means++ initialisation done")

# ── 4. GPU K-Means main loop ──────────────────────────────────────────────────
# Synchronise GPU before starting timer (important for accurate timing)
torch.cuda.synchronize()
t_start = time.perf_counter()

labels = torch.zeros(N, dtype=torch.long, device=device)

for iteration in range(1, MAX_ITER + 1):
    # ── Assignment step ───────────────────────────────────────────────────
    # Compute squared distances: ||X[i] - C[k]||^2 for all i, k
    # Using expansion: ||a-b||^2 = ||a||^2 + ||b||^2 - 2*a.b
    X_sq  = (X ** 2).sum(dim=1, keepdim=True)           # (N, 1)
    C_sq  = (centroids ** 2).sum(dim=1, keepdim=True).T  # (1, K)
    cross = X @ centroids.T                               # (N, K)
    dists = X_sq + C_sq - 2 * cross                      # (N, K)

    new_labels = dists.argmin(dim=1)                      # (N,)

    # Count reassignments
    changes = (new_labels != labels).sum().item()
    labels  = new_labels

    print(f"Iter {iteration}: {changes} reassignments")

    # ── Convergence check ─────────────────────────────────────────────────
    if changes == 0:
        print(f"Converged at iter {iteration}")
        break

    # ── Update step ───────────────────────────────────────────────────────
    new_centroids = torch.zeros(K, X.shape[1], dtype=torch.float32, device=device)
    counts        = torch.zeros(K, dtype=torch.float32, device=device)

    # Scatter-add: sum all vectors per cluster
    new_centroids.scatter_add_(0, labels.unsqueeze(1).expand(-1, X.shape[1]), X)
    counts.scatter_add_(0, labels, torch.ones(N, dtype=torch.float32, device=device))

    # Avoid division by zero (empty clusters — shouldn't happen with kmeans++)
    counts = counts.clamp(min=1)
    centroids = new_centroids / counts.unsqueeze(1)

# Synchronise GPU before stopping timer
torch.cuda.synchronize()
t_end = time.perf_counter()
gpu_time = t_end - t_start

# ── 5. Compute inertia ────────────────────────────────────────────────────────
X_sq  = (X ** 2).sum(dim=1, keepdim=True)
C_sq  = (centroids ** 2).sum(dim=1, keepdim=True).T
cross = X @ centroids.T
dists = X_sq + C_sq - 2 * cross
assigned_dists = dists.gather(1, labels.unsqueeze(1)).squeeze()
inertia = assigned_dists.sum().item()

# ── 6. Cluster sizes ──────────────────────────────────────────────────────────
cluster_sizes = [(labels == k).sum().item() for k in range(K)]

# ── 7. Print results (copy these into your report) ───────────────────────────
print("\n" + "=" * 60)
print("  GPU K-MEANS RESULTS")
print("=" * 60)
print(f"  Device         : {torch.cuda.get_device_name(0)}")
print(f"  Time           : {gpu_time:.5f} seconds  (gpu)")
print(f"  Inertia        : {inertia:.4e}")
print(f"  Cluster sizes  : {' '.join(str(s) for s in cluster_sizes)}")
print("=" * 60)

# ── 8. CSV line to paste directly into benchmark_results.csv ─────────────────
# Format matches your existing CSV: variant,ranks,threads_per_rank,total_procs,time_seconds
print("\n── Paste this line into benchmark_results.csv ──")
print(f"gpu,1,1,1,{gpu_time:.5f}")

# ── 9. Save labels to labels_gpu.bin (same format as your C++ output) ────────
labels_cpu = labels.cpu().numpy().astype(np.int32)
with open("cluster_labels_gpu.bin", "wb") as f:
    f.write(struct.pack("i", N))
    f.write(labels_cpu.tobytes())
print("\nCluster labels saved to cluster_labels_gpu.bin")
print("Download this file if you want to run evaluate.py on it locally.")

Loading vectors.bin ...
Loaded 50000 vectors, 512 dims
K=10  MAX_ITER=50  SEED=42
Device: cuda
K-Means++ initialisation ...
K-Means++ initialisation done
Iter 1: 48495 reassignments
Iter 2: 15831 reassignments
Iter 3: 8278 reassignments
Iter 4: 5786 reassignments
Iter 5: 4415 reassignments
Iter 6: 3255 reassignments
Iter 7: 2531 reassignments
Iter 8: 1888 reassignments
Iter 9: 1435 reassignments
Iter 10: 1177 reassignments
Iter 11: 1036 reassignments
Iter 12: 1024 reassignments
Iter 13: 963 reassignments
Iter 14: 936 reassignments
Iter 15: 927 reassignments
Iter 16: 924 reassignments
Iter 17: 857 reassignments
Iter 18: 792 reassignments
Iter 19: 746 reassignments
Iter 20: 659 reassignments
Iter 21: 532 reassignments
Iter 22: 504 reassignments
Iter 23: 423 reassignments
Iter 24: 359 reassignments
Iter 25: 271 reassignments
Iter 26: 228 reassignments
Iter 27: 201 reassignments
Iter 28: 171 reassignments
Iter 29: 146 reassignments
Iter 30: 120 reassignments
Iter 31: 106 reassignments
Iter

In [ ]:
"""
plot_scaling.py  —  updated version that handles the GPU row
Run after adding the gpu line to benchmark_results.csv:
    gpu,1,1,1,<time>
"""

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Load CSV ──────────────────────────────────────────────────────────────────
df = pd.read_csv("benchmark_results.csv",
                 names=["variant","ranks","threads_per_rank","total_procs","time_seconds"],
                 skiprows=1)

print(f"Loaded {len(df)} benchmark rows from benchmark_results.csv\n")
print(df.to_string(index=False))

# ── Serial baseline ───────────────────────────────────────────────────────────
serial_time = df[df.variant == "serial"]["time_seconds"].values[0]
gpu_time    = df[df.variant == "gpu"]["time_seconds"].values[0] if "gpu" in df.variant.values else None

print(f"\nSerial baseline: {serial_time:.4f}s")
if gpu_time:
    print(f"GPU time:        {gpu_time:.4f}s  (speedup: {serial_time/gpu_time:.2f}x)")

# ── Filter out serial and gpu for scaling plots ───────────────────────────────
cpu_df = df[~df.variant.isin(["serial", "gpu"])].copy()
cpu_df["speedup"]    = serial_time / cpu_df["time_seconds"]
cpu_df["efficiency"] = cpu_df["speedup"] / cpu_df["total_procs"]

# ── Colour map ────────────────────────────────────────────────────────────────
COLORS = {"omp": "#1f77b4", "mpi": "#ff7f0e", "mpi_omp": "#2ca02c"}

# ══════════════════════════════════════════════════════════════════════════════
# Plot 1: Speedup & Efficiency  (CPU only, GPU annotated as horizontal line)
# ══════════════════════════════════════════════════════════════════════════════
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Distri-Cluster: CPU Scaling + GPU Reference", fontsize=13, fontweight="bold")

for variant, grp in cpu_df.groupby("variant"):
    grp = grp.sort_values("total_procs")
    c   = COLORS.get(variant, "gray")
    ax1.plot(grp["total_procs"], grp["speedup"],  marker="o", color=c, label=variant)
    ax2.plot(grp["total_procs"], grp["efficiency"], marker="o", color=c, label=variant)

# Ideal linear speedup reference
max_p = cpu_df["total_procs"].max()
xs    = np.array([1, max_p])
ax1.plot(xs, xs, "k--", linewidth=1, label="ideal linear")

# GPU horizontal reference line on speedup plot
if gpu_time:
    gpu_speedup = serial_time / gpu_time
    ax1.axhline(gpu_speedup, color="red", linestyle=":", linewidth=1.8,
                label=f"GPU T4 ({gpu_speedup:.1f}×)")
    ax1.annotate(f"GPU T4\n{gpu_speedup:.1f}×",
                 xy=(max_p * 0.6, gpu_speedup),
                 xytext=(max_p * 0.6, gpu_speedup + 0.3),
                 color="red", fontsize=9, ha="center")

ax1.set_xlabel("Total Processors")
ax1.set_ylabel("Speedup (×)")
ax1.set_title("Speedup vs Total Processors")
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)
ax1.set_xticks(sorted(cpu_df["total_procs"].unique()))

ax2.axhline(1.0, color="k", linestyle="--", linewidth=1, label="ideal (100%)")
ax2.set_xlabel("Total Processors")
ax2.set_ylabel("Parallel Efficiency")
ax2.set_title("Efficiency vs Total Processors")
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)
ax2.set_xticks(sorted(cpu_df["total_procs"].unique()))

plt.tight_layout()
plt.savefig("speedup_efficiency.png", dpi=150, bbox_inches="tight")
print("\nSaved: speedup_efficiency.png")

# ══════════════════════════════════════════════════════════════════════════════
# Plot 2: Time comparison bar chart — all configs including GPU
# ══════════════════════════════════════════════════════════════════════════════
all_df = df.copy()

# Build labels for bars
def make_label(row):
    if row["variant"] == "serial":
        return "serial"
    if row["variant"] == "gpu":
        return "GPU T4"
    if row["variant"] == "omp":
        return f"omp {int(row['threads_per_rank'])}T"
    if row["variant"] == "mpi":
        return f"mpi {int(row['ranks'])}R"
    if row["variant"] == "mpi_omp":
        return f"mpi_omp\n{int(row['ranks'])}R×{int(row['threads_per_rank'])}T"
    return row["variant"]

all_df["label"] = all_df.apply(make_label, axis=1)

def bar_color(row):
    if row["variant"] == "serial": return "#888888"
    if row["variant"] == "gpu":    return "#d62728"
    return COLORS.get(row["variant"], "gray")

all_df["color"] = all_df.apply(bar_color, axis=1)

fig2, ax3 = plt.subplots(figsize=(14, 5))
bars = ax3.bar(range(len(all_df)), all_df["time_seconds"],
               color=all_df["color"].tolist(), edgecolor="white", linewidth=0.5)

# Value labels on bars
for bar, val in zip(bars, all_df["time_seconds"]):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f"{val:.2f}s", ha="center", va="bottom", fontsize=8)

ax3.set_xticks(range(len(all_df)))
ax3.set_xticklabels(all_df["label"].tolist(), fontsize=8)
ax3.set_ylabel("Wall-Clock Time (s)")
ax3.set_title("Distri-Cluster: Wall-Clock Time — All Configurations + GPU", fontweight="bold")
ax3.grid(True, axis="y", alpha=0.3)

# Legend
legend_patches = [
    mpatches.Patch(color="#888888", label="Serial"),
    mpatches.Patch(color=COLORS["omp"],     label="OpenMP"),
    mpatches.Patch(color=COLORS["mpi"],     label="MPI"),
    mpatches.Patch(color=COLORS["mpi_omp"], label="MPI+OMP Hybrid"),
    mpatches.Patch(color="#d62728",          label="GPU (CUDA)"),
]
ax3.legend(handles=legend_patches, fontsize=9)

plt.tight_layout()
plt.savefig("time_comparison.png", dpi=150, bbox_inches="tight")
print("Saved: time_comparison.png")

plt.show()
print("\nDone.")